# Regression Track

23CSE301 ML Capstone — Review 1

10 algorithms trained and compared on the same preprocessed dataset / held-out test set.

## 1. Dataset Loading & Audit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="colorblind")


In [ ]:
# Load the Used Cars dataset (see data/README.md for the data dictionary)
RAW_PATH = "../data/raw/autos.csv"  # adjust filename to whatever download_dataset.py produced

df = pd.read_csv(RAW_PATH, encoding="latin-1")

print("Shape:", df.shape)
df.info()
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print("\nTarget (price) distribution:")
print(df["price"].describe())

## 2. Exploratory Data Analysis

Distribution plots, correlation heatmap, target distribution, feature-target scatter plots. Add a Markdown insight note after each plot.

In [ ]:
# Distribution plots for key numeric features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df["price"], bins=50, ax=axes[0, 0]).set_title("Price distribution")
sns.histplot(df["yearOfRegistration"], bins=50, ax=axes[0, 1]).set_title("Year of registration")
sns.histplot(df["powerPS"], bins=50, ax=axes[1, 0]).set_title("Power (PS)")
sns.histplot(df["kilometer"], bins=50, ax=axes[1, 1]).set_title("Kilometers driven")
plt.tight_layout()
plt.savefig("../reports_distributions.png", bbox_inches="tight") if False else None
plt.show()

# Correlation heatmap (numeric features)
numeric_cols = ["price", "yearOfRegistration", "powerPS", "kilometer", "monthOfRegistration"]
plt.figure(figsize=(6, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

# Feature-target scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="powerPS", y="price", alpha=0.3, ax=axes[0]).set_title("Power vs. Price")
sns.scatterplot(data=df, x="kilometer", y="price", alpha=0.3, ax=axes[1]).set_title("Kilometers vs. Price")
plt.tight_layout()
plt.show()

# TODO: add a Markdown cell after each plot noting what it reveals about the data

## 3. Preprocessing & Feature Engineering

Cleaning, encoding, scaling (fit on train only), train/test split, at least one engineered feature with justification.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# --- Cleaning ---
# Drop rows with clearly invalid price / registration year (documented, justified filtering)
df_clean = df[(df["price"] > 100) & (df["price"] < 200_000)]
df_clean = df_clean[(df_clean["yearOfRegistration"] >= 1950) & (df_clean["yearOfRegistration"] <= 2026)]
df_clean = df_clean.drop_duplicates()

# Drop columns that carry no predictive signal (IDs, timestamps, near-constant)
drop_cols = ["dateCrawled", "name", "nrOfPictures", "postalCode", "dateCreated", "lastSeen", "offerType", "abtest"]
df_clean = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])

# --- Feature engineering ---
# Vehicle age at time of ad creation is more informative than raw registration year
df_clean["vehicle_age"] = 2016 - df_clean["yearOfRegistration"]  # dataset was crawled in 2016

target = "price"
numeric_features = ["vehicle_age", "powerPS", "kilometer", "monthOfRegistration"]
categorical_features = ["seller", "vehicleType", "gearbox", "model", "fuelType", "brand", "notRepairedDamage"]

X = df_clean[numeric_features + categorical_features]
y = df_clean[target]

# --- Split first, then fit scalers/encoders on train only (avoid leakage) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print("Train shape:", X_train_proc.shape, "Test shape:", X_test_proc.shape)

## 4. Model Training — All 10 Algorithms

Linear, Ridge, Lasso, ElasticNet, Polynomial, Decision Tree, Random Forest, Gradient Boosting, SVR, KNN Regressor.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso Regression": Lasso(alpha=0.1, random_state=RANDOM_STATE),
    "ElasticNet Regression": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE),
    "Polynomial Regression (deg=2)": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "SVR": SVR(kernel="rbf", C=1.0),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
}

fitted_models = {}
for name, model in models.items():
    model.fit(X_train_proc, y_train)
    fitted_models[name] = model
    print(f"Trained: {name}")

## 5. Comparative Evaluation

Summary table: R², RMSE, MAE for all 10 models, ranked by R².

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.model_selection import cross_val_score

rows = []
for name, model in fitted_models.items():
    y_pred = model.predict(X_test_proc)
    rows.append({
        "Model": name,
        "R2": r2_score(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "MAE": mean_absolute_error(y_test, y_pred),
    })

results_df = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
print(results_df)

# 5-fold cross-validated R2 for the two best-performing models
top_2 = results_df["Model"].head(2).tolist()
for name in top_2:
    cv_scores = cross_val_score(fitted_models[name], X_train_proc, y_train, cv=5, scoring="r2")
    print(f"{name}: 5-fold CV R2 = {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Q6. Decision Tree Regressor

Tune max depth; show feature importance.

In [ ]:
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import numpy as np

print("--- Q6: Decision Tree Regressor ---")
# Tune max_depth
param_grid_dt = {"max_depth": [3, 5, 7, 10, 15, None]}
grid_dt = GridSearchCV(DecisionTreeRegressor(random_state=RANDOM_STATE), param_grid_dt, cv=5, scoring="r2", n_jobs=-1)
grid_dt.fit(X_train_proc, y_train)
print("Decision Tree best params:", grid_dt.best_params_, "best R2:", grid_dt.best_score_)

# Show feature importance
best_dt = grid_dt.best_estimator_
importances = best_dt.feature_importances_
# Get feature names after preprocessing
num_features = numeric_features
cat_features = preprocessor.named_transformers_['cat'].named_steps['encode'].get_feature_names_out(categorical_features)
all_features = np.concatenate([num_features, cat_features])

indices = np.argsort(importances)[::-1][:10] # Top 10

plt.figure(figsize=(10, 6))
plt.title("Decision Tree - Top 10 Feature Importances")
plt.bar(range(10), importances[indices], align="center")
plt.xticks(range(10), all_features[indices], rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Q7. Random Forest Regressor

Ensemble baseline; tune n_estimators.

In [ ]:
print("--- Q7: Random Forest Regressor ---")
# Tune n_estimators
param_grid_rf = {"n_estimators": [50, 100, 200]}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), param_grid_rf, cv=5, scoring="r2", n_jobs=-1)
grid_rf.fit(X_train_proc, y_train)
print("Random Forest best params:", grid_rf.best_params_, "best R2:", grid_rf.best_score_)

# We can also compare it to the baseline from earlier
baseline_rf_r2 = results_df.loc[results_df["Model"] == "Random Forest Regressor", "R2"].values[0]
print(f"Improvement over default RF (test set): {baseline_rf_r2:.4f} -> (CV best) {grid_rf.best_score_:.4f}")


## 7. Visualisation

Residual plot & predicted-vs-actual for the best model; feature importance for a tree-based model.

In [ ]:
# TODO: final visualisations